# STF -> KITTI Data Pipeline Demo
**Team 3, Section 11, Topic 17** -- Extreme-Climate Multimodal Sensor Fusion and Failure Forecasting

1. **Data extraction** -- sources, collection steps, representative samples
2. **Data cleaning** -- missing values, transformations, formatting, validation
3. **Exploratory data analysis** -- statistics, plots, important findings

Every cell below calls the team's actual pipeline scripts in `baseline/scripts/` -- nothing here is reimplemented or mocked, this notebook is a thin runner + inline display on top of the real, tested code.

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

# Resolve the repo root relative to this notebook (walk up from the current
# working directory until a marker file is found), rather than a hardcoded
# path -- VS Code's Jupyter extension sets cwd to the notebook's own
# directory by default, so this works for any teammate's clone.
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "context.md").exists():
            return p
    raise FileNotFoundError("could not find repo root (looked for context.md) -- "
                             "run this notebook from somewhere inside the repo clone")

REPO = find_repo_root(Path.cwd())
SCRIPTS = REPO / "baseline" / "scripts"
KITTI_FORMAT = REPO / "baseline" / "data" / "kitti_format"

def run(args, cwd=SCRIPTS):
    """Run a real pipeline script and print its output, same as the terminal demo."""
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    print(f"[exit code: {result.returncode}]")
    return result

## 1. Data Extraction

**Source**: Seeing Through Fog (STF/DENSE), Bijelic et al. CVPR 2020 -- registration-gated, non-commercial research use (see `context.md` for full terms). ~13,500 frames across clear/fog/rain/snow, northern Europe.

**Collection steps** (all real scripts, already run -- not re-run here since the full download is 94GB / ~1hr):
1. `download_stf_subset.py` -- scoped download from a registered-access manifest
2. `baseline/scripts/01_extract_archives.py` -- extracts the lidar/label/calib archives needed for Phase 1 (pure-Python split-archive scanner, written for a Windows GPU box with no admin rights to install `zip`/7-Zip)
3. `baseline/scripts/02_convert_to_kitti.py` -- converts STF's native format into the KITTI layout OpenPCDet expects

**Representative sample** -- one real frame, before and after conversion:

In [ ]:
raw_label = REPO / "baseline/data/kitti_source/gt_labels/cam_left_labels_TMP/2019-09-11_18-41-27_02580.txt"
converted_label = KITTI_FORMAT / "training/label_2/000000.txt"

print("=== RAW STF label (27 fields, STF class names, STF object order) ===")
print("\n".join(raw_label.read_text().splitlines()[:3]))
print()
print("=== CONVERTED KITTI label (15 fields, PassengerCar->Car, DontCare moved last) ===")
print(converted_label.read_text())

## 2. Data Cleaning

What the conversion step does: class remap (`PassengerCar`->`Car`), 27->15 field truncation, lidar channel drop (5->4 fields), ROS tf-tree calibration -> flat KITTI `calib.txt`, 16-bit raw image -> 8-bit PNG, and a label-ordering fix (a real bug the team found by actually running training, not caught by file-level checks: real objects must precede `DontCare` objects in each label file, or OpenPCDet's own info-generation code silently misreads which box belongs to which object).

### 2a. Automated validation -- clean run

In [ ]:
run(["python", "05_verify_conversion.py", "--kitti-format", str(KITTI_FORMAT)])

### 2b. The same check on deliberately corrupted input

This script used to **silently pass** when data was missing -- a real bug, found and fixed. Now every check fails loudly instead.

In [ ]:
calib_file = KITTI_FORMAT / "training/calib/000000.txt"
calib_backup = calib_file.with_suffix(".txt.bak")

# safe to re-run: restores first if a previous run left the backup in place
if calib_backup.exists() and not calib_file.exists():
    calib_backup.rename(calib_file)

calib_file.rename(calib_backup)
try:
    run(["python", "05_verify_conversion.py", "--kitti-format", str(KITTI_FORMAT)])
finally:
    calib_backup.rename(calib_file)
    print(f"restored: {calib_file.name}")

## 3. Exploratory Data Analysis

Real statistics and plots computed from the converted data (sampling real lidar frames, not synthesizing anything) by `baseline/scripts/06_data_statistics.py`.

In [ ]:
eda_dir = REPO / "baseline/data/eda"
run(["python", "06_data_statistics.py", "--kitti-format", str(KITTI_FORMAT), "--out", str(eda_dir)])

In [ ]:
print((eda_dir / "summary.txt").read_text())

In [ ]:
for name in ["A_class_distribution_by_split.png", "B_objects_per_frame.png",
             "C_dimensions_by_class.png", "D_distance_by_class.png",
             "E_lidar_points_per_frame.png"]:
    display(Image(filename=str(eda_dir / name)))

### Key findings
- Pedestrian outnumbers Car in the train split alone, but Car leads once all 5 splits are combined -- worth checking for train/test distribution shift
- 76.8% of train frames contain zero Cyclist instances -- the class is real but sparse
- 33.8% of train-split rows carry STF's sentinel `-1.00` dims (3D box not measured) -- excluded from the dimension/distance plots rather than left in to skew them
- Lidar points/frame is sharply bimodal (~55k vs ~110k, not a smooth spread) -- **open question, not yet explained**, flagged honestly rather than hidden